# UTS Data Science - Exploratory Data Analysis, Representativitas, dan Bias Sampel

Notebook ini membahas bagian **Data Understanding** yang dinilai pada UTS:

1. kualitas data (missing value, duplikasi, dan outlier),
2. exploratory data analysis (distribusi dan korelasi),
3. representativitas sampel acak, dan
4. potensi bias serta keterbatasan data.

> **Batasan UTS:** notebook ini tidak melakukan data preparation untuk modeling, training model, evaluasi model, atau deployment.

In [ ]:
# Jika library belum tersedia, jalankan satu kali lalu restart kernel:
# %pip install pandas numpy matplotlib seaborn scipy

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
SAMPLE_SIZE = 100_000  # wajib berada pada rentang 10.000-100.000
MIN_SAMPLE_SIZE = 10_000
MAX_SAMPLE_SIZE = 100_000

## 1. Memuat data

Pencarian path dibuat fleksibel agar notebook dapat dijalankan dari folder `notebooks`, root proyek, atau Google Colab.

In [ ]:
candidate_paths = [
    Path("../archive/creditcard.csv"),
    Path("archive/creditcard.csv"),
    Path("creditcard.csv"),
    Path("/content/creditcard.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "creditcard.csv tidak ditemukan. Letakkan file di folder archive/ "
        "atau ubah candidate_paths pada cell ini."
    )

df = pd.read_csv(data_path)
required_columns = {"Time", "Amount", "Class"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

print(f"Path data : {data_path.resolve()}")
print(f"Ukuran data: {df.shape[0]:,} baris x {df.shape[1]:,} kolom")
display(df.head())

## 2. Data dictionary ringkas

Karena `V1`-`V28` telah dianonimkan, makna bisnis dan unit aslinya tidak dapat diinterpretasikan langsung. Hal ini penting dicatat sebagai keterbatasan data dan keterbatasan audit fairness.

In [ ]:
data_dictionary = pd.DataFrame([
    {"Variabel": "Time", "Tipe": "Numerik kontinu", "Unit": "Detik", "Deskripsi": "Waktu relatif sejak transaksi pertama pada dataset."},
    {"Variabel": "V1-V28", "Tipe": "Numerik kontinu", "Unit": "Tidak tersedia", "Deskripsi": "Fitur anonim hasil transformasi; arti bisnis asli tidak tersedia."},
    {"Variabel": "Amount", "Tipe": "Numerik kontinu", "Unit": "Tidak dinyatakan pada file", "Deskripsi": "Nilai transaksi. Unit mata uang harus diverifikasi dari sumber asli."},
    {"Variabel": "Class", "Tipe": "Biner/kategorikal", "Unit": "0 atau 1", "Deskripsi": "0 = normal, 1 = fraud."},
])
display(data_dictionary)

## 3. Audit kualitas data

Audit ini memeriksa tipe data, missing value, duplikasi baris, nilai tak hingga, dan ringkasan statistik. Duplikasi **tidak langsung dihapus** pada tahap Data Understanding karena baris yang sama belum tentu merupakan kesalahan tanpa ID transaksi dan definisi bisnis.

In [ ]:
quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "unique_n": df.nunique(dropna=False),
})

duplicate_n = int(df.duplicated().sum())
infinite_n = int(np.isinf(df.select_dtypes(include="number")).sum().sum())

print(f"Total missing value : {int(df.isna().sum().sum()):,}")
print(f"Baris duplikat      : {duplicate_n:,} ({duplicate_n / len(df) * 100:.3f}%)")
print(f"Nilai tak hingga   : {infinite_n:,}")
display(quality_summary)
display(df.describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).T)

In [ ]:
def iqr_outlier_summary(frame, columns):
    records = []
    for column in columns:
        q1, q3 = frame[column].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = (frame[column] < lower) | (frame[column] > upper)
        records.append({
            "variabel": column, "batas_bawah": lower, "batas_atas": upper,
            "outlier_n": int(mask.sum()), "outlier_pct": mask.mean() * 100,
        })
    return pd.DataFrame(records).sort_values("outlier_pct", ascending=False)

outlier_table = iqr_outlier_summary(df, ["Time", "Amount"])
display(outlier_table)
print("Catatan: outlier adalah observasi ekstrem secara statistik, bukan otomatis data salah.")

## 4. Desain random sampling

Metode utama adalah **simple random sampling tanpa pengembalian** dengan seed tetap agar reproducible. Ukuran sampel default adalah 100.000 baris. Data penuh hanya dipakai sebagai acuan untuk menguji representativitas; EDA utama menggunakan sampel.

Alasan memilih batas atas 100.000: label fraud sangat langka, sehingga sampel 10.000 diperkirakan hanya memuat sekitar belasan transaksi fraud dan memberikan estimasi kelas minoritas yang kurang stabil.

In [ ]:
if not MIN_SAMPLE_SIZE <= SAMPLE_SIZE <= MAX_SAMPLE_SIZE:
    raise ValueError("SAMPLE_SIZE harus berada pada rentang 10.000-100.000.")
if SAMPLE_SIZE > len(df):
    raise ValueError("SAMPLE_SIZE tidak boleh melebihi jumlah baris data.")

sample_df = (
    df.sample(n=SAMPLE_SIZE, replace=False, random_state=RANDOM_STATE)
      .sort_index()
      .reset_index(drop=True)
)

population_fraud_rate = df["Class"].mean()
expected_fraud = SAMPLE_SIZE * population_fraud_rate
inclusion_probability = SAMPLE_SIZE / len(df)

print(f"Ukuran sampel           : {len(sample_df):,}")
print(f"Seed                    : {RANDOM_STATE}")
print(f"Peluang inklusi per baris: {inclusion_probability:.2%}")
print(f"Ekspektasi jumlah fraud : {expected_fraud:.1f}")
display(sample_df.head())

In [ ]:
# Stabilitas jumlah fraud pada beberapa ukuran sampel, diulang dengan 100 seed.
candidate_sizes = [10_000, 25_000, 50_000, 100_000]
stability_records = []

for size in candidate_sizes:
    if size > len(df):
        continue
    fraud_counts = np.array([
        int(df["Class"].sample(n=size, replace=False, random_state=seed).sum())
        for seed in range(100)
    ])
    stability_records.append({
        "sample_size": size,
        "fraud_mean": fraud_counts.mean(),
        "fraud_std": fraud_counts.std(ddof=1),
        "fraud_min": fraud_counts.min(),
        "fraud_max": fraud_counts.max(),
        "relative_std_pct": fraud_counts.std(ddof=1) / fraud_counts.mean() * 100,
    })

stability_table = pd.DataFrame(stability_records)
display(stability_table)

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(
    stability_table["sample_size"], stability_table["fraud_mean"],
    yerr=stability_table["fraud_std"], marker="o", capsize=5
)
ax.set(title="Stabilitas Jumlah Fraud pada 100 Pengambilan Sampel",
       xlabel="Ukuran sampel", ylabel="Jumlah fraud (rata-rata +/- SD)")
plt.show()

## 5. Exploratory Data Analysis

EDA berikut menggunakan `sample_df`. Fokusnya adalah ketidakseimbangan label, distribusi nilai transaksi dan waktu, perbedaan antar-kelas, serta fitur yang paling berhubungan dengan label fraud.

In [ ]:
class_summary = (
    sample_df["Class"].value_counts().sort_index().rename("jumlah").to_frame()
)
class_summary["persentase"] = class_summary["jumlah"] / len(sample_df) * 100
class_summary.index = class_summary.index.map({0: "Normal (0)", 1: "Fraud (1)"})
display(class_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=sample_df, x="Class", ax=axes[0], color="#4C78A8")
axes[0].set(title="Distribusi Kelas (Skala Linear)", xlabel="Class", ylabel="Jumlah")
sns.countplot(data=sample_df, x="Class", ax=axes[1], color="#F58518")
axes[1].set_yscale("log")
axes[1].set(title="Distribusi Kelas (Skala Log)", xlabel="Class", ylabel="Jumlah (log)")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = sample_df.assign(
    TimeHour=sample_df["Time"] / 3600,
    LogAmount=np.log1p(sample_df["Amount"]),
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(plot_df["Amount"], bins=80, ax=axes[0, 0], color="#4C78A8")
axes[0, 0].set(title="Distribusi Amount", xlabel="Amount")
axes[0, 0].set_xlim(0, plot_df["Amount"].quantile(0.99))

sns.histplot(plot_df["LogAmount"], bins=80, ax=axes[0, 1], color="#59A14F")
axes[0, 1].set(title="Distribusi log(1 + Amount)", xlabel="log(1 + Amount)")

sns.histplot(data=plot_df, x="TimeHour", hue="Class", bins=48,
             stat="density", common_norm=False, element="step", ax=axes[1, 0])
axes[1, 0].set(title="Distribusi Waktu Menurut Kelas", xlabel="Waktu relatif (jam)")

sns.boxplot(data=plot_df, x="Class", y="LogAmount", ax=axes[1, 1])
axes[1, 1].set(title="Amount Menurut Kelas", xlabel="Class", ylabel="log(1 + Amount)")

plt.tight_layout()
plt.show()

display(sample_df.groupby("Class")[["Time", "Amount"]].agg(["count", "mean", "median", "std"]))

In [ ]:
numeric_columns = sample_df.select_dtypes(include="number").columns
correlation = sample_df[numeric_columns].corr(method="spearman")
target_corr = correlation["Class"].drop("Class").sort_values(key=np.abs, ascending=False)
top_features = target_corr.head(12).index.tolist()

display(target_corr.head(15).rename("spearman_corr_with_class").to_frame())

plt.figure(figsize=(10, 8))
sns.heatmap(sample_df[top_features + ["Class"]].corr(method="spearman"),
            cmap="vlag", center=0, square=False)
plt.title("Korelasi Spearman: Fitur Teratas dan Class")
plt.tight_layout()
plt.show()

print("Catatan: korelasi menunjukkan hubungan, bukan sebab-akibat.")

## 6. Uji representativitas sampel

Sampel dibandingkan dengan data penuh melalui:

- selisih proporsi kelas fraud,
- interval kepercayaan 95% untuk proporsi fraud,
- **standardized mean difference (SMD)** untuk setiap variabel numerik, dan
- statistik Kolmogorov-Smirnov (KS) untuk perbedaan distribusi.

Pedoman praktis: `|SMD| < 0,10` menunjukkan perbedaan kecil. P-value KS tidak dipakai sendirian karena sangat sensitif terhadap ukuran data yang besar.

In [ ]:
def wilson_interval(successes, n, z=1.96):
    proportion = successes / n
    denominator = 1 + z**2 / n
    center = (proportion + z**2 / (2 * n)) / denominator
    margin = z * math.sqrt(proportion * (1 - proportion) / n + z**2 / (4 * n**2)) / denominator
    return center - margin, center + margin

population_rate = df["Class"].mean()
sample_fraud_n = int(sample_df["Class"].sum())
sample_rate = sample_df["Class"].mean()
ci_low, ci_high = wilson_interval(sample_fraud_n, len(sample_df))

class_representation = pd.DataFrame({
    "metrik": ["Jumlah fraud", "Proporsi fraud", "Proporsi fraud (%)"],
    "populasi": [int(df["Class"].sum()), population_rate, population_rate * 100],
    "sampel": [sample_fraud_n, sample_rate, sample_rate * 100],
})
display(class_representation)
print(f"95% CI proporsi fraud sampel: [{ci_low:.6f}, {ci_high:.6f}]")
print(f"Proporsi populasi berada di dalam CI: {ci_low <= population_rate <= ci_high}")
print(f"Selisih absolut proporsi: {abs(sample_rate - population_rate):.6f}")

In [ ]:
def standardized_mean_difference(population, sample):
    pooled_sd = math.sqrt((population.var(ddof=1) + sample.var(ddof=1)) / 2)
    return 0.0 if pooled_sd == 0 else (sample.mean() - population.mean()) / pooled_sd

representation_records = []
for column in df.select_dtypes(include="number").columns:
    population_values = df[column].dropna()
    sample_values = sample_df[column].dropna()
    ks_result = ks_2samp(population_values, sample_values, alternative="two-sided")
    representation_records.append({
        "variabel": column,
        "mean_populasi": population_values.mean(),
        "mean_sampel": sample_values.mean(),
        "median_populasi": population_values.median(),
        "median_sampel": sample_values.median(),
        "SMD": standardized_mean_difference(population_values, sample_values),
        "KS_statistic": ks_result.statistic,
        "KS_pvalue": ks_result.pvalue,
    })

representation_table = pd.DataFrame(representation_records)
representation_table["abs_SMD"] = representation_table["SMD"].abs()
representation_table = representation_table.sort_values("abs_SMD", ascending=False)
display(representation_table)

max_abs_smd = representation_table["abs_SMD"].max()
share_small_smd = representation_table["abs_SMD"].lt(0.10).mean()
print(f"SMD absolut maksimum : {max_abs_smd:.4f}")
print(f"Variabel dengan |SMD| < 0,10: {share_small_smd:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_rep = representation_table.sort_values("SMD")
colors = np.where(plot_rep["abs_SMD"] < 0.10, "#59A14F", "#E45756")
axes[0].barh(plot_rep["variabel"], plot_rep["SMD"], color=colors)
axes[0].axvline(-0.10, color="black", linestyle="--", linewidth=1)
axes[0].axvline(0.10, color="black", linestyle="--", linewidth=1)
axes[0].set(title="Standardized Mean Difference", xlabel="SMD: sampel vs populasi")

top_ks = representation_table.nlargest(12, "KS_statistic").sort_values("KS_statistic")
axes[1].barh(top_ks["variabel"], top_ks["KS_statistic"], color="#4C78A8")
axes[1].set(title="12 Statistik KS Terbesar", xlabel="KS statistic")
plt.tight_layout()
plt.show()

## 7. Analisis bias dan keterbatasan

### Bias yang perlu dilaporkan

1. **Class imbalance:** fraud adalah kelas sangat langka. Akurasi biasa akan menyesatkan jika kelak dipakai untuk modeling.
2. **Sampling variance:** simple random sampling tidak menjamin jumlah fraud yang persis sama pada setiap seed. Sampel 100.000 mengurangi, tetapi tidak menghilangkan, variasi ini.
3. **Coverage bias:** dataset hanya merepresentasikan transaksi yang masuk ke sistem pengumpulan data sumber pada periode dan populasi tertentu. Generalisasi ke institusi, wilayah, waktu, atau pola transaksi lain harus dibatasi.
4. **Feature anonymization:** `V1`-`V28` tidak memiliki makna bisnis yang tersedia, sehingga penjelasan penyebab dan audit fairness terhadap kelompok sensitif tidak dapat dilakukan.
5. **Label bias:** kualitas analisis bergantung pada proses penetapan label fraud. Transaksi fraud yang tidak terdeteksi dapat tercatat sebagai normal.
6. **Duplicate uncertainty:** baris identik tidak otomatis duplikat pencatatan karena dataset tidak menyediakan ID transaksi. Penghapusan harus menunggu justifikasi pada fase Data Preparation.
7. **Lisensi dan provenance:** sumber asli, lisensi, periode, populasi, dan metode anonimisasi harus diverifikasi dan disitasi dari halaman penyedia sebelum laporan dikumpulkan.

### Mitigasi yang disarankan

- gunakan seed tetap dan laporkan metode serta ukuran sampel;
- bandingkan sampel dengan data penuh menggunakan proporsi kelas, SMD, dan bentuk distribusi;
- pertahankan simple random sample untuk estimasi distribusi populasi;
- jika analisis khusus kelas fraud diperlukan, buat sampel stratified **terpisah** dan jangan menganggap proporsi kelasnya mewakili populasi;
- pada tahap modeling nanti, gunakan split yang mempertahankan kelas dan metrik seperti precision, recall, F1, PR-AUC, serta biaya kesalahan.

## 8. Kesimpulan otomatis siap diadaptasi ke laporan

Jalankan cell berikut setelah semua analisis selesai. Angka dihasilkan langsung dari data dan sampel, sehingga tidak perlu disalin secara manual.

In [ ]:
representation_label = (
    "baik pada indikator numerik yang diuji"
    if max_abs_smd < 0.10 and ci_low <= population_rate <= ci_high
    else "belum sepenuhnya baik pada seluruh indikator yang diuji"
)

display(Markdown(f"""
### Ringkasan hasil

Dataset terdiri dari **{len(df):,} baris dan {df.shape[1]} variabel**. Pemeriksaan kualitas menemukan
**{int(df.isna().sum().sum()):,} missing value** dan **{duplicate_n:,} baris identik**. Label fraud pada data penuh berjumlah
**{int(df['Class'].sum()):,} transaksi ({population_rate * 100:.4f}%)**, sehingga terdapat ketidakseimbangan kelas yang ekstrem.

Simple random sampling tanpa pengembalian menghasilkan **{len(sample_df):,} baris** dengan seed **{RANDOM_STATE}**.
Sampel memuat **{sample_fraud_n:,} transaksi fraud ({sample_rate * 100:.4f}%)**. Interval kepercayaan Wilson 95% untuk
proporsi fraud sampel adalah **{ci_low * 100:.4f}% sampai {ci_high * 100:.4f}%**. SMD absolut maksimum sebesar
**{max_abs_smd:.4f}**, dan **{share_small_smd:.1%}** variabel memiliki |SMD| < 0,10. Berdasarkan indikator tersebut,
representativitas sampel dinilai **{representation_label}**.

Walaupun sampel cukup representatif secara statistik, generalisasi substantif tetap dibatasi oleh kelangkaan fraud,
anonimisasi fitur, ketidakjelasan atribut kelompok sensitif, potensi kesalahan label, serta cakupan populasi dan periode
yang harus diverifikasi dari sumber asli.
"""))

## 9. Checklist sebelum pengumpulan UTS

- [ ] URL sumber asli dan tanggal akses dicantumkan.
- [ ] Lisensi/izin penggunaan sudah diverifikasi.
- [ ] Data dictionary disesuaikan dengan dokumentasi sumber.
- [ ] Semua cell berhasil dijalankan dari awal sampai akhir.
- [ ] Grafik diberi interpretasi dalam bahasa sendiri pada laporan.
- [ ] Angka pada laporan konsisten dengan output notebook.
- [ ] Keterbatasan dan bias tidak dihapus meskipun hasil representativitas baik.
- [ ] Notebook dan laporan tidak mengklaim causalitas dari korelasi.